# Week 4 Lecture: A Python Review for DevOps, Part 2
**NET2008 DevOps · Algonquin College**

Week 3 covered Python fundamentals: values, collections, control flow, and functions. This week focuses on writing scripts that handle failures, work with files, and grow into reusable tools.

Everything for this session lives in this notebook. Read the prose, run the code cells, and experiment with the examples. Several cells raise exceptions on purpose so you can practice reading the errors.

## What We Will Review

11. **Exceptions and error handling**
    - Syntax errors, runtime exceptions, logic errors, and tracebacks
    - The exception hierarchy
    - `try`, `except`, `else`, and `finally`
    - Catching specific or multiple exceptions
    - Raising
    - Custom exceptions
    - Fail fast, retry, graceful degradation, logging, and cleanup

12. **File handling**
    - Opening files and choosing file modes
    - Reading, writing, and iterating over files
    - Binary files and text encoding
    - Working with paths using `pathlib`

13. **Modules and packages**
    - Importing modules, import styles, and aliases
    - Creating your own modules and packages
    - The role of `__init__.py`
    - The `if __name__ == "__main__":` pattern
    - Module search paths and the standard library

14. **Object-oriented programming**
    - Classes, objects, attributes, and methods
    - Constructors, instance variables, and class variables
    - Instance, class, and static methods
    - Encapsulation and properties
    - Inheritance
    - Abstract classes and magic methods

Some examples create files in the notebook's working directory (`devices.txt`, `report.txt`, `devices.py`, `inventory/`, and `toolkit/`). Inspect them in the file browser after running those cells.


---
## 11. Exceptions and error handling

### 11.1 What an exception is

There are three ways a program can be wrong, and they fail at different moments.

- A **syntax error** means the file is not valid Python. Nothing runs at all — the interpreter rejects the file before executing the first line.
- A **runtime error** means the code is valid Python, but a specific line could not do what it was asked with the values it was actually given. Python signals this by **raising an exception**.
- A **logic error** means the code runs perfectly and produces the wrong answer. No exception, no traceback, just a quietly wrong number. These are the expensive ones.

An exception is an object. When a line fails, Python creates an instance of an exception class, stops executing the current block, and looks for a handler up the call stack. If nobody handles it, the program dies and prints a traceback.

**Read a traceback from the bottom up.** The last line is the exception type and message — the *what*. The lines above it are the call stack — the *where*, most recent call last.

In [ ]:
# 1. Syntax error: Python cannot understand the code.
def bad_code():
  compile('print("Hello"', "<example>", "exec")

try:
    bad_code()
except SyntaxError as error:
    print(f"Syntax error: {error.msg}")

# 2. Runtime error: Valid code fails while running.
try:
    print(10 / 0)
except ZeroDivisionError as error:
    print("Runtime error:", error)

# 3. Logic error: Code runs, but the answer is wrong.
price = 100
discount = 20
final_price = price + discount  # Bug: should subtract

print(f"final price is {final_price} which is a logic error")
print(f"Expected final price: {price - discount}")

Syntax error: '(' was never closed
Runtime error: division by zero
final price is 120 which is a logic error
Expected final price: 80


### 11.1.1 The exception hierarchy

Python organizes exceptions like a family tree. A parent exception describes a broad category; its children describe more specific problems.

```
BaseException
├── KeyboardInterrupt          # User presses Ctrl+C
├── SystemExit                 # Program requests an exit
└── Exception                  # Ordinary program errors
    ├── ArithmeticError
    │   └── ZeroDivisionError
    ├── LookupError
    │   ├── KeyError
    │   └── IndexError
    ├── OSError
    │   └── FileNotFoundError
    ├── ValueError
    └── TypeError
```

`KeyboardInterrupt` (you pressing Ctrl+C) and `SystemExit` deliberately sit *outside* `Exception`. That is why `except Exception:` is safe and a bare `except:` is not — the bare form swallows your Ctrl+C too.

In [ ]:
try:
    10 / 0
except ZeroDivisionError as error:
    print("ZeroDivisionError:", error)

try:
    {"host": "sw01"}["port"]
except KeyError as error:
    print("KeyError:", error)

try:
    [10, 20][5]
except IndexError as error:
    print("IndexError:", error)

try:
    int("forty two")
except ValueError as error:
    print("ValueError:", error)

try:
    int(["42"])
except TypeError as error:
    print("TypeError:", error)

try:
    open("does-not-exist.conf")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

ZeroDivisionError: division by zero
KeyError: 'port'
IndexError: list index out of range
ValueError: invalid literal for int() with base 10: 'forty two'
TypeError: int() argument must be a string, a bytes-like object or a real number, not 'list'
FileNotFoundError: [Errno 2] No such file or directory: 'does-not-exist.conf'


| Exception | Typical cause in DevOps code |
|-----------|------------------------------|
| `KeyError` | a config/JSON payload is missing a field you assumed |
| `IndexError` | parsing a line that had fewer columns than expected |
| `ValueError` | `int("443/tcp")` — right type, impossible value |
| `TypeError` | `"port " + 443` — wrong type entirely |
| `FileNotFoundError` | the inventory file is not where the script expected |
| `PermissionError` | running without the rights to read/write that path |
| `TimeoutError` / `ConnectionError` | the device or API did not answer |
| `ZeroDivisionError` | averaging an empty list of samples |

### 11.2 `try`

`try` marks a block of code as risky. Nothing about `try` changes how the code behaves when it succeeds — it only gives Python somewhere to jump to when something raises.

Keep the `try` block **as small as you can**. A `try` wrapped around thirty lines tells you something in there failed; a `try` around the one line that talks to the network tells you exactly what failed.

In [ ]:
config = {"hostname": "sw01"}

try:
    port = config["port"]        # this raises
    print("never reached")
except KeyError as error:
    print("handled: no 'port' in config")

print("execution continues normally")

handled: no 'port' in config
execution continues normally


### 11.3 `except`

`except` names the exception type (or types) it is willing to handle, and optionally binds the exception object to a name with `as`.

That object is worth looking at rather than discarding. `str(e)` gives the message, `type(e)` gives the type, `e.errorno` give the error number, etc.

In [ ]:
try:
    open("does-not-exist.conf")
except FileNotFoundError as e:
    print("message   :", str(e))
    print("type      :", type(e))
    print("errno     :", e.errno)
    print("filename  :", e.filename)

type      : <class 'FileNotFoundError'>
message   : [Errno 2] No such file or directory: 'does-not-exist.conf'
errno     : 2
filename  : does-not-exist.conf


### 11.4 `else`

An `else` block runs **only if the `try` block succeeds**. It looks redundant at first — why not just put those lines at the end of the `try`? Because anything inside `try` is also protected by your `except`, and you rarely want that.

If the "on success" work lives inside `try`, an unrelated `KeyError` from your reporting code gets caught by the handler written for the risky call, and now your log says "device unreachable" when the device was fine and your formatting was broken. `else` keeps the two apart.

In [ ]:
def average(samples):
    try:
        result = sum(samples) / len(samples)
    except ZeroDivisionError:
        print("  no samples to average")
        return None
    else:
        # runs only when the division succeeded
        print(f"  average computed: {result}")
        return result

print("with data:");  average([11.4, 13.2, 9.8])
print("empty:");      average([])

with data:
  average computed: 11.466666666666667
empty:
  no samples to average


In [ ]:
try:
    try:
        port = int("8080")          # Succeeds
    except ValueError:
        print("Invalid port number")
    else:
        print("Port parsed:", port)
        print(10 / 0)               # New error in else
except ZeroDivisionError:
    print("Caught the error raised in else")

Port parsed: 8080
Caught the error raised in else


### 11.5 `finally`

A `finally` block runs **no matter what** — after a successful `try`, after a handled exception, after an unhandled one on its way up the stack, and even after a `return`. It is where cleanup goes: close the file, release the lock, log out of the session, delete the temp directory.

In [ ]:
def demo(mode):
    try:
        if mode == "raise":
            raise ValueError("raised error")
        print("runs first in try")
        return "normal"
    except ValueError as e:
        return f"handled {e}"
    finally:
        # runs in all three paths, including the two that return
        print(f"[{mode}] runs in finally")

for mode in ["normal", "return", "raise"]:
    print(f"    mode -> {demo(mode)}\n")

runs first in try
[normal] runs in finally
    mode -> normal

runs first in try
[return] runs in finally
    mode -> normal

[raise] runs in finally
    mode -> handled raised error



> **Common pitfall.** Cleanup written *after* the `try/except` instead of inside `finally` is skipped entirely if an exception escapes the block. `finally` is the only placement that survives every exit path.

### 11.6 Multiple exceptions

Two forms, and they mean different things.

- **One handler, several types:** `except (KeyError, IndexError) as e:` — use when the response is identical for all of them.
- **Several handlers, stacked:** a separate `except` per type — use when the response differs.

Python checks handlers top to bottom and runs the **first** one that matches. The rest are skipped, even if they would also match.

In [ ]:
def read_port(config):
    try:
        print(f'config["port"]: {config["port"]}')
        return int(config["port"])
    except (KeyError, TypeError):          # one response for two failure modes
        return 22                           # a sane default
    except ValueError:                      # a different response
        print(f"  'port' is present but not a number: {config['port']!r}")
        return None

print(read_port({"port": "443"}))     # 443
print(read_port({}))                   # 22   - KeyError  -> default
print(read_port({"port": None}))       # 22   - TypeError -> default
print(read_port({"port": "443/tcp"}))  # None - ValueError

config["port"]: 443
443
22
config["port"]: None
22
config["port"]: 443/tcp
  'port' is present but not a number: '443/tcp'
None


### 11.7 Catching specific exceptions

Order your handlers **most specific first**. Because a parent class matches all of its children, putting `except Exception:` above `except FileNotFoundError:` means the specific handler is unreachable — Python takes the first match, and the parent always matches first.

Three rules that will save you hours:

1. **Never write a bare `except:`.** It catches `KeyboardInterrupt` and `SystemExit` as well, so your script becomes impossible to stop with Ctrl+C, and it hides typos — a misspelled variable raises `NameError`, which a bare `except` happily swallows.
2. **Catch the narrowest type that describes the problem you actually know how to handle.** If you cannot do anything sensible about an exception, do not catch it.
3. **`except Exception:` is acceptable at the top level** of a long-running job — a loop over 500 devices should not die because one of them raised something you did not predict — but log the exception when you do it.

In [ ]:
# Why a bare `except` is a trap: this hides a simple typo as if it were an expected failure.
def buggy():
    try:
        total = sum([1, 2, 3])
        print(totl)            # typo -> NameError
    except Exception as error:  # Broad handler matches first
        print("Broad handler:", type(error).__name__, error)
    except NameError as error:   # Never reached
        print("Specific handler:", error)

buggy()

Broad handler: NameError name 'totl' is not defined


In [ ]:
# Why a bare `except` is a trap: this hides a simple typo as if it were an expected failure.
def buggy():
    try:
        total = sum([1, 2, 3])
        print(totl)            # typo -> NameError
    except NameError as error:   # Specific error should be first
        print("Specific handler:", error)
    except Exception as error:  # Broad handler matches first
        print("Broad handler:", type(error).__name__, error)

buggy()

Specific handler: name 'totl' is not defined


### 11.8 Raising exceptions

`raise` is how *your* code reports a problem to its caller. Use it when a function is handed something it cannot work with. Returning `None` or `-1` instead pushes the problem downstream, where it shows up later as a confusing `TypeError` in unrelated code.

- `raise ValueError("message")` — raise a new exception.
- `raise` on its own, inside an `except` block — re-raise the exception currently being handled, preserving its original traceback. This is the right way to log something and let it continue up the stack.

In [ ]:
def set_port(port):
    if not isinstance(port, int):
        raise TypeError(f"port must be an int, got {type(port).__name__}")
    if not 0 < port <= 65535:
        raise ValueError(f"port {port} is outside 1-65535")
    return port

for candidate in [443, 70000, "443"]:
    try:
        print("ok:", set_port(candidate))
    except (TypeError, ValueError) as e:
        print(f"rejected: {type(e).__name__}: {e}")

ok: 443
rejected: ValueError: port 70000 is outside 1-65535
rejected: TypeError: port must be an int, got str


In [ ]:
def risky():
    raise ConnectionError("device did not answer")

def with_logging():
    try:
        risky()
    except ConnectionError:
        print("  logging this, then passing it along")
        raise          # bare re-raise: same exception, original traceback intact

try:
    with_logging()
except ConnectionError as e:
    print("caller still sees:", e)

  logging this, then passing it along
caller still sees: device did not answer


### 11.9 Custom exceptions

A custom exception is an error type you name for your application. Create it by defining a class that inherits from Exception:

In [ ]:
class DeviceOfflineError(Exception):
    """Raised when a device cannot be reached."""


def connect(hostname):
    if hostname == "sw02":
        raise DeviceOfflineError(f"{hostname} is offline")
    return f"Connected to {hostname}"


try:
    print(connect("sw02"))
except DeviceOfflineError as error:
    print("Connection failed:", error)
else:
    print("Connection succeeded")

Connection failed: sw02 is offline


### 11.10 Error-handling strategies

Catching an exception is the easy half. Deciding **what to do next** is the design decision, and there are five standard answers. Choosing the wrong one is how an automation script ends up either fragile (dies on the first hiccup) or dangerous (carries on with garbage data).

| Strategy | Use it when | Danger if misapplied |
|----------|-------------|----------------------|
| Fail fast | The input is wrong and continuing would corrupt something | Downtime over a harmless blip |
| Retry | The failure is transient — network, rate limit, lock contention | Hammering a service that is genuinely down |
| Graceful degradation | One item of many failed and the rest are still worth processing | Silently reporting partial results as complete |
| Logging errors | Always, alongside whichever of the others you picked | Logs nobody reads; secrets written to disk |
| Cleanup after failure | Any time you hold a resource: file, socket, lock, temp dir | Leaked handles, stale locks, half-written files |

**Fail fast.** Validate at the boundary and raise immediately. The cheapest bug to fix is the one that stops the program one line after the bad value arrives, not three functions later.

In [ ]:
def send_message(address, message):
    if not address:
        raise ValueError("Address is required")

    print(f"Sending '{message}' to {address}")


try:
    send_message("", "Hello")
except ValueError as error:
    print("Stopped:", error)

Stopped: Address is required


**Retry.** Only for *transient* failures, and always bounded: a maximum number of attempts and a growing delay between them (exponential backoff), so a struggling service gets room to recover instead of a tighter flood of requests. Never retry something that is not idempotent unless you know it is safe to run twice.

In [ ]:
attempts = 0

def check_device():
    global attempts
    attempts += 1
    print(f"Attempt {attempts}")

    if attempts == 1:
        raise ConnectionError("Device did not respond")

    return "Device is online"


for attempt in range(2):
    try:
        result = check_device()
        print(result)
        break  # Success: stop retrying
    except ConnectionError as error:
        print("Failed:", error)

Attempt 1
Failed: Device did not respond
Attempt 2
Device is online


**Graceful degradation.** When you are processing a collection, one bad element should not discard the other 499. Substitute a safe default, mark the result as incomplete, and carry on — but make the gap **visible** in the output. A report that silently omits the devices it could not reach is worse than one that crashed.

In [ ]:
devices = ["sw01", "sw02", "sw03"]

for device in devices:
    try:
        if device == "sw02":
            raise ConnectionError("No response")

        print(f"{device}: online")
    except ConnectionError:
        print(f"{device}: status unavailable")

print("Device check complete")

sw01: online
sw02: status unavailable
sw03: online
Device check complete


**Logging errors.** A log records what went wrong so you can investigate later. Use logging instead of print() in scripts that run without someone watching them.

In [ ]:
import logging

logging.basicConfig(level=logging.ERROR, format="%(levelname)s: %(message)s")

try:
    port = int("not-a-number")
except ValueError as error:
    logging.error(f"Could not read the port: {error}")

print("Program continues")

02:08:24 ERROR    root: Could not read the port: invalid literal for int() with base 10: 'not-a-number'


Program continues


> **Common pitfall.** Never log credentials, tokens, or full request bodies. "Log everything" becomes a secret-management incident the first time someone ships those logs to a third-party aggregator.

**Cleanup after failure.** Any resource you acquire must be released on every exit path. `finally` does this; a context manager (`with`) does it for you and is the preferred form — which is exactly where section 12 picks up.

In [ ]:
class Session:
    '''Stand-in for an SSH/API session that must always be closed.'''

    def __init__(self, host):
        self.host = host
        print(f"  opened session to {host}")

    def run(self, command):
        raise ConnectionError("link dropped mid-command")

    def close(self):
        print(f"  closed session to {self.host}")


session = Session("sw01")
try:
    session.run("show version")
except ConnectionError as e:
    print(f"  command failed: {e}")
finally:
    session.close()        # runs whether the command worked or not

  opened session to sw01
  command failed: link dropped mid-command
  closed session to sw01


---
## 12. File handling

Configs, inventories, logs, reports, state files, CI artifacts — the substrate of DevOps work is files. This section covers the built-in file API, then `pathlib`, which is how you should be writing new code.

### 12.1 Opening files

`open()` returns a **file object** — a handle you read from or write to, then close. Three arguments matter:

```python
open(path, mode="r", encoding=None)
```

- **path** — relative paths resolve against the *current working directory* of the process, not the folder the script lives in. That difference is why a script works when you run it by hand and fails under cron.
- **mode** — what you intend to do with it (section 12.2).
- **encoding** — for text files, specify an encoding such as encoding="utf-8" so the result does not depend on the system’s default encoding. Binary mode ("rb", "wb", etc.) does not use an encoding.

Every open file holds an OS-level handle. Leak enough of them and the process hits its file-descriptor limit and starts failing in ways that look unrelated. Close what you open — or better, let `with` do it (section 12.5).

In [ ]:
import os
from pathlib import Path

print("current working directory:", os.getcwd())

for item in Path.cwd().iterdir():
    print(item.name)


f = open("devices.txt", mode="w", encoding="utf-8")     # creates the file
f.write("sw01,10.0.0.1,48\n")
f.write("sw02,10.0.0.2,24\n")
f.write("rtr-edge-01,10.0.0.254,8\n")
f.close()                                           # explicit, and easy to forget

print(f"closed? {f.closed}")

current working directory: /content
.config
devices.txt
sample_data
closed? True




```
# This is formatted as code
```

### 12.2 File modes

| Mode | Meaning | If the file exists | If it does not |
|------|---------|--------------------|----------------|
| `r` | read (default) | reads from the start | `FileNotFoundError` |
| `w` | write | **truncates it to zero bytes (erase)** | creates it |
| `a` | append | writes at the end | creates it |
| `x` | exclusive create and write | `FileExistsError` | creates it |
| `b` | binary modifier (`rb`, `wb`, `ab`) | — | — |
| `+` | read *and* write modifier (`r+`, `w+`, `a+`) | — | — |

Two combinations worth knowing: `r+` opens for read and write **without** truncating, while `w+` truncates first. And `x` is the safe way to write a file only if you are certain you are not overwriting someone else's — it is an atomic check-and-create, unlike testing `exists()` first and then opening, which has a race condition between the two calls.

In [ ]:
# `x` refuses to clobber an existing file.
try:
    open("devices.txt", "x", encoding="utf-8")
except FileExistsError as e:
    print("exclusive create refused:", e.filename)

with open("devices.txt") as file:
    print(file.read())

# `a` adds to the end; `w` would have wiped the file.
with open("devices.txt", "a", encoding="utf-8") as f:
    f.write("ap-lobby-01,10.0.1.8,2\n")

with open("devices.txt", "r", encoding="utf-8") as file:
    print(file.read())


exclusive create refused: devices.txt
sw01,10.0.0.1,48
sw02,10.0.0.2,24
rtr-edge-01,10.0.0.254,8
ap-lobby-01,10.0.1.7,2

sw01,10.0.0.1,48
sw02,10.0.0.2,24
rtr-edge-01,10.0.0.254,8
ap-lobby-01,10.0.1.7,2
ap-lobby-01,10.0.1.7,2



> **Common pitfall.** Opening a file you meant to read with `"w"` destroys it instantly and silently — there is no confirmation and no undo. When you only intend to read, pass `"r"` explicitly rather than relying on the default, so the intent is visible in the code review.

### 12.3 Reading: `read()`, `readline()`, `readlines()`

| Call | Returns | Memory |
|------|---------|--------|
| `f.read()` | the entire file as one `str` | whole file in RAM |
| `f.read(n)` | at most `n` characters | bounded |
| `f.readline()` | the next single line, including its `\n` | one line |
| `f.readlines()` | a `list` of every line, each including `\n` | whole file in RAM |

A file object keeps a **position**. Each read continues from where the last one stopped, which is why calling `read()` twice gives you the content and then an empty string. `f.seek(0)` rewinds.

For a 40 GB log, `read()` and `readlines()` will exhaust memory. section 12.6 covers the form you should reach for by default.

In [ ]:
# Open the file for reading. "r" is the default mode.
# The with block closes the file automatically when it ends.
with open("devices.txt", encoding="utf-8") as f:
    # Read up to 12 characters from the start of the file.
    # repr() makes invisible characters like \n visible in the output.
    print("read(12)   :", repr(f.read(12)))

    # Read from the current position through the end of that line.
    # This does not necessarily read a whole line, since we may be
    # partway through it after read(12).
    print("readline() :", repr(f.readline()))

    # Show the current position in the file.
    # In text mode, treat this value as a position marker for seek(),
    # not necessarily as a character count.
    print("tell()     :", f.tell())

    # Move back to the beginning of the file.
    f.seek(0)
    print("read():", f.read())

    # Read all lines into a list of strings, including any \n characters.
    f.seek(0)
    print("readlines():", f.readlines())

    # readlines() reached the end, so another read returns an empty string.
    print("read() now :", repr(f.read()))

read(12)   : 'sw01,10.0.0.'
readline() : '1,48\n'
tell()     : 17
read(): sw01,10.0.0.1,48
sw02,10.0.0.2,24
rtr-edge-01,10.0.0.254,8
ap-lobby-01,10.0.1.7,2
ap-lobby-01,10.0.1.7,2

readlines(): ['sw01,10.0.0.1,48\n', 'sw02,10.0.0.2,24\n', 'rtr-edge-01,10.0.0.254,8\n', 'ap-lobby-01,10.0.1.7,2\n', 'ap-lobby-01,10.0.1.7,2\n']
read() now : ''


### 12.4 Writing: `write()` and `writelines()`

`f.write(s)` writes the string and returns the number of characters written. It adds **nothing** — no newline, no separator. If you want line breaks, you write them.

`f.writelines(list_of_strings)` is not the counterpart to `readlines()` you might expect: it also adds no separators, so it is really "write each of these strings end to end".

In [ ]:
rows = ["sw01,up", "sw02,down", "rtr-edge-01,up"]

with open("report.txt", "w", encoding="utf-8") as f:
    n = f.write("hostname,status\n")
    print("characters written by that call:", n)
    for row in rows:
        f.write(f"{row}\n") # Write each row followed by a newline

print(open("report.txt", encoding="utf-8").read())


characters written by that call: 16
hostname,status
sw01,up
sw02,down
rtr-edge-01,up



In [ ]:
# Forgetting the separator: everything lands on one line.
with open("broken.txt", "w", encoding="utf-8") as f:
    f.writelines(rows)          # no newlines anywhere

print(repr(open("broken.txt", encoding="utf-8").read()))

'sw01,upsw02,downrtr-edge-01,up'


### 12.5 Iterating over a file

For most line-based files, loop over the file directly:

```python
with open(path, encoding="utf-8") as f:
    for line in f:
        ...
```
Python reads lines as the loop needs them, so it does not load the whole file into memory. A line normally includes its ending \n; rstrip("\n") removes that newline while preserving other whitespace. A single very long line will still need memory for that line.

Use enumerate() when an error message should identify the line:

```python
with open(path, encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        print(f"Line {line_number}: {line.rstrip(chr(10))}")
```

In [ ]:
# First, create a small file.
with open("devices.txt", "w", encoding="utf-8") as f:
    f.write("switch\nrouter\nfirewall\n")

# Then read it one line at a time.
with open("devices.txt", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        name = line.rstrip("\n")  # Remove the newline at the end.
        print(f"Line {line_number}: {name}")

Line 1: switch
Line 2: router
Line 3: firewall


### 12.6 Binary files

Add b to a file mode to work with bytes instead of text (for example, "rb" reads bytes and "wb" writes bytes). Python does not decode the contents or change line endings.

Use binary mode for images, ZIP files, and other files that are not plain text. Also use it when you need the file’s exact bytes, such as when calculating a checksum.

```python
with open("photo.jpg", "rb") as f:
    data = f.read()

print(data[:4])  # The first four bytes
```

In [ ]:
payload = b"\x7fELF\x02\x01\x01\x00" + bytes(range(8))

with open("firmware.bin", "wb") as f:
    f.write(payload)

with open("firmware.bin", "rb") as f:
    header = f.read(4)
    rest = f.read()

print("header      :", header)
print("as hex      :", header.hex())
print("rest length :", len(rest), "bytes")

header      : b'\x7fELF'
as hex      : 7f454c46
rest length : 12 bytes


### 12.8 Working with paths: `pathlib`

The old way of handling paths was string manipulation plus a pile of functions from `os` and `os.path`. `pathlib` replaces all of it with one object that knows it is a path.

A `Path` composes with the `/` operator, works across operating systems (it renders `\` on Windows and `/` elsewhere), and carries the filesystem operations as methods.

| Member | What it does |
|--------|--------------|
| `Path("a") / "b" / "c.txt"` | join path segments |
| `.exists()` | does anything exist at this path |
| `.is_file()` / `.is_dir()` | and is it a file / a directory |
| `.mkdir(parents=True, exist_ok=True)` | create a directory, parents included, no error if present |
| `.glob("*.txt")` | matching entries in **this** directory |
| `.rglob("*.txt")` | matching entries **recursively**, at any depth |
| `.read_text(encoding=...)` / `.write_text(s, encoding=...)` | whole-file read/write, open and close included |
| `.read_bytes()` / `.write_bytes(b)` | the binary equivalents |
| `.name` `.stem` `.suffix` `.parent` | filename, filename without extension, extension, containing directory |
| `.resolve()` | absolute, symlink-free form |
| `.unlink()` / `.rmdir()` | delete a file / an empty directory |

In [ ]:
from pathlib import Path

inventory_dir = Path("inventory") # stores the path "inventory" in a variable.
inventory_dir.mkdir(parents=True, exist_ok=True)      # safe to run every time

report=inventory_dir / "report.txt"                  # no string concatenation
report.write_text("hostname,status\nsw01,up\n", encoding="utf-8")

print("path      :", report)
print("absolute  :", report.resolve())
print("exists    :", report.exists())
print(f"is_file:", report.is_file())
print("name      :", report.name)
print("stem:", report.stem)
print("suffix:", report.suffix)
print("parent    :", report.parent)
print("is_dir:", report.parent.is_dir())
print("contents  :", repr(report.read_text(encoding="utf-8")))

path      : inventory/report.txt
absolute  : /content/inventory/report.txt
exists    : True
is_file: True
name      : report.txt
stem: report
suffix: .txt
parent    : inventory
is_dir: True
contents  : 'hostname,status\nsw01,up\n'


In [ ]:
from pathlib import Path

folder = Path("inventory")
(folder / "old").mkdir(parents=True, exist_ok=True)

(folder / "new.txt").write_text("new device\n", encoding="utf-8")
(folder / "old" / "old.txt").write_text("old device\n", encoding="utf-8")

# Look only inside inventory.
for p in folder.glob("*.txt"):
  print("glob:", p.name)

# Look inside inventory and its subfolders.
print("rglob:", [str(p) for p in folder.rglob("*.txt")])

glob: report.txt
glob: new.txt
rglob: ['inventory/report.txt', 'inventory/new.txt', 'inventory/old/old.txt', 'inventory/archive/2026-09-01.txt']


`mkdir()` deserves its two keyword arguments. Without `exist_ok=True` it raises `FileExistsError` the second time your script runs, which is the single most common reason a working script fails on its second execution. Without `parents=True` it refuses to create `a/b/c` unless `a/b` already exists.

In [ ]:
deep = Path("inventory") / "2026" / "09" / "22"

deep.mkdir(parents=True, exist_ok=True)     # creates all three levels
deep.mkdir(parents=True, exist_ok=True)     # second run: no error

try:
    deep.mkdir()                             # neither flag: raises
except FileExistsError as e:
    print("without exist_ok ->", type(e).__name__)

without exist_ok -> FileExistsError


---
## 13. Modules and packages

A module is a `.py` file. A package is a directory of modules. That is the entire concept — everything else is mechanics around how Python finds them and what happens when it loads them. This section is the point where a script becomes a tool.

### 13.1 Importing modules

`import x` does three things, in order:

1. **Finds** `x` by walking the module search path.
2. **Executes** the file, top to bottom, exactly once per interpreter session.
3. **Binds** the resulting module object to the name `x` in your namespace, and caches it in `sys.modules`.

Step 2 is the one people forget. Importing a module *runs* it — which is why anything a module does at import time (opening a connection, reading a file, printing) happens to everyone who imports it, whether they wanted it or not. Step 3 is why editing a module does not affect an already-running session.

In [ ]:
import sys
import json                      # standard library

print("module object :", json)
print("cached?       :", "json" in sys.modules)

module object : <module 'json' from '/usr/lib/python3.13/json/__init__.py'>
cached?       : True


### 13.2 Import styles

| Style | You then write | Use when |
|-------|----------------|----------|
| `import json` | `json.loads(...)` | the default; the prefix says where the name came from |
| `import os.path` | `os.path.join(...)` | importing a submodule |
| `from json import loads` | `loads(...)` | you use one or two names heavily |
| `from json import loads, dumps` | both, unprefixed | same, for a short explicit list |
| `from json import *` | everything, unprefixed | **avoid** |

`from module import *` copies every public name into your namespace. Two problems: you can no longer tell where a name came from when reading the code, and a later star-import silently overwrites names from an earlier one. The bug it produces looks like a function changing behaviour for no reason.

Plain `import json` is the safe default. The prefix is three characters and it makes the origin of every call obvious.

`as` renames an import in your namespace. Three legitimate reasons to use it:

- Convention — whole ecosystems expect import pandas as pd, import numpy as np. Follow them; deviating makes your code harder to read, not more original.
- Collision — two modules with the same leaf name, or a module whose name shadows a local variable.
- Length — import xml.etree.ElementTree as ET is genuinely better than typing the full path at every call site.

In [ ]:
import json
import json as j                        # alias
from json import loads                   # one function
from json import loads as parse_json     # one funation, renamed

payload = '{"hostname": "sw01", "ports": 48}'
print(json.loads(payload))
print(j.loads(payload))
print(loads(payload))
print(parse_json(payload))

{'hostname': 'sw01', 'ports': 48}
{'hostname': 'sw01', 'ports': 48}
{'hostname': 'sw01', 'ports': 48}
{'hostname': 'sw01', 'ports': 48}


### 13.4 Creating your own modules

Any `.py` file next to your script is importable by its filename without the extension. The cell below writes a real module to disk; the one after imports it.

The `if __name__ == "__main__":` block at the bottom is explained *later* — for now, notice that importing the module does not print anything.

In [ ]:
%%writefile devices.py
'''Small helper module for NET2008 device work.'''

DEFAULT_TIMEOUT = 5


def average_ping(*samples):
    '''Return the mean of the given ping samples.'''
    if not samples:
        raise ValueError("at least one sample is required")
    return sum(samples) / len(samples)


def grade_ping(avg, threshold=15.0):
    '''Classify an average latency as "good" or "slow".'''
    return "good" if avg < threshold else "slow"


def _internal_helper():
    '''Leading underscore: not part of the public API.'''
    return "implementation detail"


if __name__ == "__main__":
    print("running devices.py directly")
    print(grade_ping(average_ping(11.4, 13.2, 9.8)))

Writing devices.py


In [ ]:
import devices
from devices import average_ping as amir

avg = devices.average_ping(11.4, 13.2, 9.8)
avg_amir = amir(11.4, 13.2, 9.8)
print(avg, devices.grade_ping(avg))
print(avg_amir)
print("constant  :", devices.DEFAULT_TIMEOUT)
print("docstring :", devices.__doc__)

11.466666666666667 good
11.466666666666667
constant  : 5
docstring : Small helper module for NET2008 device work.


> **Common pitfall.** Edit `devices.py` and re-run the `import devices` cell and nothing changes — the module is already in `sys.modules`, so the `import` is a no-op. Either restart the kernel or force a reload:
>
> ```python
> import importlib; importlib.reload(devices)
> ```
>
> This only bites in long-lived sessions such as notebooks and REPLs. A script that runs and exits always gets fresh code.

In [ ]:
import importlib

importlib.reload(devices)     # picks up edits to devices.py without restarting the kernel
print("reloaded:", devices.average_ping(10, 20))

reloaded: 15.0


### 13.5 Packages

A package is a directory containing modules, imported with dotted names: `my_tools.greetings`. It is how you group related modules once one file is no longer enough.

```
my_tools/
├── __init__.py
└── greetings.py
```

Packages nest: `my_tools/greetings.py` is `my_tools.greetings`, as long as each level has its own `__init__.py`.

### 13.6 `__init__.py`

`__init__.py` is what makes a directory a package in the traditional sense. It runs when the package is first imported, and it has one genuinely useful job: **defining the package's public surface**. Without it, users of your package have to know your internal file layout (`from my_tools.greetings impor hello`).

`__all__` is a list of names that `from my_tool imports *` will export. It is also the clearest available statement of "this is the supported API, everything else is internal".

In [ ]:
from pathlib import Path

Path("my_tools").mkdir(exist_ok=True)

In [ ]:
%%writefile my_tools/__init__.py
"""Simple greeting tools."""

from .greetings import hello

print("my_tools package loaded")

Overwriting my_tools/__init__.py


In [ ]:
%%writefile my_tools/greetings.py
def hello(name):
    return f"Hello, {name}!"

Writing my_tools/greetings.py


In [ ]:
import importlib

importlib.invalidate_caches()  # Let Python find the newly created package.

from my_tools.greetings import hello as greet

print(greet("Amir"))

Hello, Amir!


### 13.7 `__name__`

Every module has a `__name__` attribute. Python sets it to:

- `"__main__"` when the file is **run directly** (`python greetings.py`), or
- the module's own name (`"greetings"`) when it is **imported**.

Hence the guard:

```python
if __name__ == "__main__":
    main()
```

Everything inside it runs only on direct execution. This is what lets one file be both an importable library and a runnable command-line tool — without it, importing your script would execute its whole body as a side effect.

The cell below proves both halves: the import printed nothing earlier, and running the same file as a subprocess does print.

In [ ]:
%%writefile my_tools/greetings.py
def hello(name):
    return f"Hello, {name}!"

def main():
    print(hello("Amir"))

if __name__ == "__main__":
    main()

Overwriting my_tools/greetings.py


In [ ]:
import subprocess
import sys

print("Inside this notebook, __name__ is:", __name__)

# Import the module itself so we can inspect its __name__.
import my_tools.greetings as greetings
print("When imported, greetings.__name__ is:", greetings.__name__)

# Run the file directly in a separate Python process.
result = subprocess.run(
    [sys.executable, "my_tools/greetings.py"],
    capture_output=True,
    text=True,
    check=True,
)
print("When run directly, it prints:", result.stdout.strip())

Inside this notebook, __name__ is: __main__
When imported, greetings.__name__ is: my_tools.greetings
When run directly, it prints: Hello, Amir!


### 13.8 The standard library

Python ships with a large standard library — "batteries included". Before adding a dependency, check whether the stdlib already covers it: a stdlib module needs no install step, no version pinning, no supply-chain review, and it is present on every machine your code lands on.

| Module | Use |
|--------|-----|
| `os`, `sys` | environment variables, process arguments, interpreter state |
| `pathlib` | filesystem paths |
| `json`, `csv`, `configparser` | data formats you will meet weekly |
| `logging` | structured, levelled output (section 11.11) |
| `datetime`, `time` | timestamps, durations, scheduling |
| `subprocess` | running shell commands and capturing their output |
| `argparse` | command-line interfaces for your scripts |
| `re` | regular expressions, for log and output parsing |
| `collections` | `defaultdict`, `Counter`, `namedtuple` |
| `itertools`, `functools` | iteration helpers; `wraps`, `lru_cache`, `reduce` |
| `hashlib`, `secrets` | checksums; cryptographically secure tokens |
| `tempfile`, `shutil` | scratch files and directories; copy/move/delete trees |
| `unittest` | the built-in test framework |

Outside the stdlib, the two you will install constantly in this field are `requests` (HTTP) and `PyYAML` (YAML).

In [ ]:
from collections import Counter
from datetime import datetime, timedelta
import tempfile

statuses = ["up", "down", "up", "up", "unreachable", "down"]
print("counts   :", Counter(statuses).most_common())

now = datetime(2026, 9, 22, 9, 15)
print("window   :", now.isoformat())

with tempfile.TemporaryDirectory() as tmp:      # a context manager that deletes itself
    print("scratch  :", tmp)

counts   : [('up', 3), ('down', 2), ('unreachable', 1)]
window   : 2026-09-22T09:15:00
scratch  : /tmp/tmpajarvbbd


---
## 14. Object-oriented programming

You have already been using objects all term: a string is an object with methods, so is a list, a file handle, a `Path`. This section is about defining your own.

The practical reason to care: every library you import is written this way. `requests.Session`, `pathlib.Path`, `logging.Logger`, and expect you to know what `self`, inheritance, and a dunder method are. The second reason is that a dictionary stops scaling the moment the data needs behaviour attached to it.

### 14.1 Classes

A class is a **blueprint**. It describes what every object of that kind will know (attributes) and what it will be able to do (methods). Defining a class creates no data — it creates a type.

Convention: class names are `CamelCase`, everything else stays `lower_snake_case`.

In [2]:
class Device:
    '''A network device in the inventory.'''


print(Device)                 # a class object
print(type(Device).__name__)            # classes are themselves objects, of type `type`
print(Device.__doc__)

<class '__main__.Device'>
type
A network device in the inventory.


### 14.2 Objects

An object (an **instance**) is one concrete thing built from the blueprint. Calling the class creates one. Each instance is independent and has its own identity.

In [3]:
a = Device()
b = Device()

print(a)                      # default display: class name and memory address
print(type(a).__name__)
print("same class? ", type(a) is type(b))
print("same object?", a is b)                 # False - two separate instances
print("isinstance :", isinstance(a, Device))

Device
same class?  True
same object? False
isinstance : True


### 14.3 Attributes

An attribute is a value attached to an object, reached with a dot. Python lets you add them at any time, though in real code you set them in the constructor (14.5) so every instance is guaranteed to have them.

In [9]:
a.hostname = "sw01"            # attached after the fact
a.ip = "10.0.0.1"

print("vars     :", vars(a))

print("a hostname attribute", a.hostname)
print("a hostname attribute", getattr(a, "hostname"))
print("a ip attribute", a.ip)
print("a ip attribute", getattr(a, "ip"))

print("a has attribute hostname:", hasattr(a, "hostname"))
print("b has attribute hostname:", hasattr(b, "hostname"))   # b never got one
print("getattr  :", getattr(a, "ports", "unset"))                   # default instead of AttributeError
setattr(a, "ports", 48)
print("after set:", vars(a))

vars     : {'hostname': 'sw01', 'ip': '10.0.0.1', 'ports': 48}
a hostname attribute sw01
a hostname attribute sw01
a ip attribute 10.0.0.1
a ip attribute 10.0.0.1
a has attribute hostname: True
b has attribute hostname: False
getattr  : 48
after set: {'hostname': 'sw01', 'ip': '10.0.0.1', 'ports': 48}


### 14.4 Methods

A method is a function defined inside a class. Its first parameter is the instance itself, named `self` by convention. You never pass `self` explicitly — `a.summary()` is Python's shorthand for `Device.summary(a)`.

In [11]:
class Device:
    def summary(self):
        return f"{self.hostname} ({self.ip})"


d = Device()
d.hostname, d.ip = "sw02", "10.0.0.2"

print(d.summary())               # the bound form
print(Device.summary(d))         # exactly what the line above does

sw02 (10.0.0.2)
sw02 (10.0.0.2)


### 14.5 Constructors: `__init__()`

`__init__` runs automatically right after an instance is created, and its job is to put the object into a valid initial state. Arguments passed to `Device(...)` are forwarded to it. Strictly, `__init__` is an *initialiser*, not a constructor — `__new__` allocates the object and `__init__` fills it in. You will almost never touch `__new__`.

In [20]:
class Device:
    def __init__(self, label, hostname, ip, ports=24):
        if not hostname:
            raise ValueError("hostname is required")
        self.label = label
        self.hostname = hostname
        self.ip = ip
        self.ports = ports              # default makes the third argument optional

    def summary(self):
        return f"{self.label} attributes are {self.hostname} ({self.ip}), {self.ports} ports"


dev_a = Device("a", "sw01", "10.0.0.1", 48)
dev_b = Device("b", "sw02", "10.0.0.2") # ports defaults to 24
print(dev_a.summary())
print(dev_b.summary())

try:
    dev_c = Device("c", "", "10.0.0.3")
except ValueError as e:
    print("rejected at construction:", e)

a attributes are sw01 (10.0.0.1), 48 ports
b attributes are sw02 (10.0.0.2), 24 ports
rejected at construction: hostname is required


### 14.6 Instance and Class variables

An instance variable belongs to **one object**. It is created by assigning to `self.something`, normally in `__init__`. Two instances never share them.

A class variable is defined in the class body, outside any method, and is **shared by every instance**. Use it for constants and for state that genuinely belongs to the type as a whole — a counter of how many instances exist, a registry, a default.

Lookup goes instance first, then class. So reading `self.x` finds the class variable when the instance has none — but *assigning* `self.x = ...` always creates an instance variable that shadows it from then on.

In [24]:
class Device:
    device_count = 0                  # class variable: one copy, shared default
    vendor = "generic"                 # a shared default

    def __init__(self, hostname):
        self.hostname = hostname       # instance variable: one per object
        Device.device_count += 1       # update through the CLASS, not through self


x = Device("sw01")
y = Device("sw02")

print("device count:", Device.device_count)
print("device count via instance x:", x.device_count)
print("device count via instance y:", y.device_count)
print("both see the shared default:", x.vendor, y.vendor)

x.vendor = "cisco"                     # creates an INSTANCE variable on x only
y.vendor = "juniper"                   # creates an INSTANCE variable on y only
print("after x.vendor = 'cisco' ->", x.vendor, y.vendor, Device.vendor)
print("x's own attributes:", vars(x))
print("y's own attributes:", vars(y))

device count: 2
device count via instance x: 2
device count via instance y: 2
both see the shared default: generic generic
after x.vendor = 'cisco' -> cisco juniper generic
x's own attributes: {'hostname': 'sw01', 'vendor': 'cisco'}
y's own attributes: {'hostname': 'sw02', 'vendor': 'juniper'}


> **Common pitfall — the mutable class variable.** A list or dict in the class body is shared by every instance, so appending through one instance is visible through all of them. This is one of the most common bugs in beginner Python, and it looks like magic when it happens. Give each instance its own by creating it in `__init__`.

In [25]:
class Broken:
    tags = []                 # ONE list, shared by every instance

class Fixed:
    def __init__(self):
        self.tags = []        # a new list per instance


p, q = Broken(), Broken()
p.tags.append("core")
print("Broken :", p.tags, q.tags, "  <- q sees it too")

r, s = Fixed(), Fixed()
r.tags.append("core")
print("Fixed  :", r.tags, s.tags)

Broken : ['core'] ['core']   <- q sees it too
Fixed  : ['core'] []


### 14.7 Instance, Class, and Static methods

Instance method is the default kind: they take `self` and operate on one object's data. Everything in 14.4 and 14.5 was an instance method.

A class method receives the **class** as its first parameter, named `cls`, instead of an instance. Its dominant use is the **alternate constructor**: a second way to build an instance, from a config line, a dict, a JSON payload, an API response.

Building with `cls(...)` rather than the hard-coded class name matters for subclasses — `Router.from_config_line(...)` then produces a `Router`, not a `Device`.

A static method takes neither `self` nor `cls`. It is a plain function that lives inside the class because it belongs there conceptually — a validator, a converter, a small piece of arithmetic related to the type but needing no access to any particular object.

In [27]:
class Device:
    def __init__(self, hostname, ip):
        if not self.is_valid_ip(ip):
            raise ValueError(f"bad IP: {ip!r}")
        self.hostname = hostname
        self.ip = ip

    def summary(self):                              # instance method
        return f"{self.hostname} ({self.ip})"

    @classmethod
    def from_config_line(cls, line):                # alternate constructor
        hostname, ip = (part.strip() for part in line.split(","))
        if not cls.is_valid_ip(ip):
            raise ValueError(f"bad IP in config line: {line!r}")
        return cls(hostname, ip)                    # cls, not Device

    @staticmethod
    def is_valid_ip(candidate):                     # no self, no cls
        parts = candidate.split(".")
        return len(parts) == 4 and all(p.isdigit() and 0 <= int(p) <= 255 for p in parts)


d = Device.from_config_line("sw01, 10.0.0.1")
print(d.summary())
print("validator, callable without an instance:", Device.is_valid_ip("10.0.0.300"))

try:
    Device.from_config_line("sw02, 999.1.1.1")
except ValueError as e:
    print("rejected:", e)

sw01 (10.0.0.1)
validator, callable without an instance: False
rejected: bad IP in config line: 'sw02, 999.1.1.1'


### 14.8 Encapsulation
Encapulation means letting an object manage its own data. For example, a Device should check an IP address before accepting it. That way, other code cannot accidentally give it an invalid IP through the normal interface.
Python uses naming conventions to show which attributes are intended for outside use:
- name is public. Other code can use it.
- _name is internal. Other code can access it, but the underscore says, “This is an implementation detail; avoid changing it directly.”
- __name is name-mangled. Python changes its internal name to reduce accidental clashes in subclasses. It is not a security feature.

A property lets callers use an attribute normally while the class checks changes to it:

In [33]:
class Device:
    def __init__(self, hostname, ip):
        self.hostname = hostname        # public
        self._session = None             # internal by convention
        self.__secret = "do-not-touch"   # name-mangled
        self.ip = ip                      # goes through the property setter below

    @property
    def ip(self):
        return self._ip

    @ip.setter
    def ip(self, value):
        if value.count(".") != 3:
            raise ValueError(f"not an IPv4 address: {value!r}")
        self._ip = value


d = Device("sw01", "10.0.0.1")
print("reads like an attribute:", d.ip)

d.ip = "10.0.0.9"                        # runs the setter
print("after assignment       :", d.ip)

try:
    d.ip = "not-an-ip"                   # validation on plain assignment
except ValueError as e:
    print("setter rejected it    :", e)

print("mangled name           :", d._Device__secret)
print("instance dict          :", vars(d))

reads like an attribute: 10.0.0.1
after assignment       : 10.0.0.9
setter rejected it    : not an IPv4 address: 'not-an-ip'
mangled name           : do-not-touch
instance dict          : {'hostname': 'sw01', '_session': None, '_Device__secret': 'do-not-touch', '_ip': '10.0.0.9'}


### 14.9 Inheritance

A subclass inherits every attribute and method of its parent and can add or change whatever it needs. Write it as `class Child(Parent):`.

`super()` calls the parent's version of a method. In `__init__`, calling `super().__init__(...)` is how the parent's setup runs — skip it and the attributes the parent would have created simply do not exist.

In [36]:
class Device:
    def __init__(self, hostname):
        self.hostname = hostname

    def summary(self):
        return f"Device: {self.hostname}"


class Router(Device):
    def route(self):
        return f"{self.hostname} is routing traffic"


router = Router("r01")
print(router.summary())  # Device: r01
print(router.route())    # r01 is routing traffic

a = Device("a")
print(a.summary())
try:
  print(a.route())
except AttributeError as e:
  print("missing method:", e)

Device: r01
r01 is routing traffic
Device: a
missing method: 'Device' object has no attribute 'route'


In [38]:
class Device:
    def __init__(self, hostname, ip):
        self.hostname = hostname
        self.ip = ip

    def summary(self):
        return f"{self.hostname} ({self.ip})"

    def reboot(self):
        return f"rebooting {self.hostname}"


class Router(Device):
    def __init__(self, hostname, ip, routing_protocol):
        super().__init__(hostname, ip)          # the parent's setup
        self.routing_protocol = routing_protocol


r = Router("rtr-edge-01", "10.0.0.254", "BGP")
print(r.summary())                 # inherited unchanged
print(r.reboot())                   # inherited unchanged
print(r.routing_protocol)            # added by the subclass

rtr-edge-01 (10.0.0.254)
rebooting rtr-edge-01
BGP


### 14.10 Method overriding

A subclass can replace a method by defining one with the same name. Calling it on an instance of the subclass runs the subclass's version — the parent's is shadowed, not deleted, and `super()` still reaches it.

The common pattern is **extending** rather than replacing: call `super().method()` for the shared part, then add the specialised part.

In [40]:
class Switch(Device):
    def __init__(self, hostname, ip, port_count):
        super().__init__(hostname, ip)
        self.port_count = port_count

    def summary(self):                                  # override
        return f"{super().summary()} - {self.port_count} ports"   # extend, do not duplicate


print(Switch("sw01", "10.0.0.1", 48).summary())

sw01 (10.0.0.1) - 48 ports


### 14.11 Abstract classes

An abstract class is a starting point for other classes. It says which methods every child class must provide.
Use `ABC` to make the class abstract, and `@abstractmethod` to mark a required method. You cannot create an object from the abstract class itself. You also cannot create an object from a child class until it implements every required method.

Use an abstract class when several related classes must all provide the same method, and you want Python to catch omissions early, instead of discovering the problem later when your program tries to call it. An abstract class can also contain regular methods that child classes inherit.

In [43]:
from abc import ABC, abstractmethod

class Device(ABC):
    @abstractmethod
    def status(self):
        pass


class Switch(Device):
    def status(self):
        return "Switch is online"


class Router(Device):
    pass  # Forgot to define status()


switch = Switch()
print(switch.status())  # Switch is online

try:
  router = Router()        # TypeError: Router is missing status()
except TypeError as e:
  print(e)

Switch is online
Can't instantiate abstract class Router without an implementation for abstract method 'status'


### 14.17 Magic (dunder) methods

Dunder methods are methods with double underscores before and after their names, such as `__len__`. They tell Python how your objects should behave with familiar operations.

| Method | Triggered by | Should return |
|--------|--------------|---------------|
| `__init__` | `Device(...)` | `None` (it initialises in place) |
| `__str__` | `print(x)`, `str(x)` | a readable string for humans |
| `__repr__` | the REPL, `repr(x)`, containers | an unambiguous string, ideally valid Python |
| `__len__` | `len(x)` | a non-negative `int` |
| `__eq__` | `x == y` | `True` / `False` / `NotImplemented` |
| `__lt__` | `x < y`, `sorted()`, `min()`, `max()` | `True` / `False` / `NotImplemented` |

Two rules worth memorising. First, if you define only one of `__str__`/`__repr__`, define `__repr__` — Python falls back to it when `__str__` is absent, but never the other way round. Second, defining `__eq__` sets `__hash__` to `None`, making instances unhashable and unusable in a `set` or as dict keys; define `__hash__` too if you need that.

In [51]:
class Device:
    def __init__(self, hostname, ports):
        self.hostname = hostname
        self.ports = ports

    def __str__(self):
        return f"{self.hostname}: {self.ports} ports"

    def __repr__(self):
        return f"Device({self.hostname!r}, {self.ports!r})"

    def __len__(self):
        return self.ports

    def __eq__(self, other):
        if not isinstance(other, Device):
            return NotImplemented
        return (self.hostname, self.ports) == (other.hostname, other.ports)


sw1 = Device("sw01", 24)
sw2 = Device("sw02", 48)

print(sw1)         # sw01: 24 ports
print(repr(sw1))   # Device('sw01', 24)
print([sw1])       # [Device('sw01', 24)]
print(len(sw1))    # 24
print(sw1 == sw1)  # Calls sw1.__eq__(sw2) to check sw2 is an instance of Device

sw01: 24 ports
Device('sw01', 24)
[Device('sw01', 24)]
24
True


---
## Wrap-up

Week 3 was about expressing an idea in Python. This session was about making that idea hold up unattended:

- **Exceptions** let a script distinguish between a problem it can recover from and one it should stop for — and section 11.11's five strategies are the actual decision you make every time you write an `except`.
- **File handling** and `pathlib` are how state survives between runs. `with` is not optional style; it is the only form that closes the file when things go wrong.
- **Modules and packages** are the step from a script to a tool. `if __name__ == "__main__":` is what lets one file be both.
- **OOP** is the vocabulary every library you import is written in, and the right answer once data needs behaviour attached to it. Prefer composition; use inheritance when the relationship is genuinely is-a.

Three habits to carry forward:

1. **Never write `except: pass`.** If you catch something, handle it, log it, or re-raise it. A silently swallowed exception is a bug that will be found by a user rather than by you.
2. **Let the failure name itself.** A `DeviceUnreachableError` in a traceback tells the on-call engineer where to look; a bare `KeyError` on line 214 does not.
3. **When something surprises you, inspect it.** `type(x)`, `vars(x)`, `sys.path`, and `func.__name__` answer most "why is it doing that?" questions in one line.

**This week's lab:** `W04_Lab_RobustDeviceToolkit.ipynb` applies all five sections to one problem — a device toolkit with its own module, a class hierarchy, custom exceptions, JSON persistence via `pathlib`, and decorators for retry and logging.